<center><h2><strong><font color="blue"> Data Engineering for Data Science - Week 04</font></strong></h2></center>

<center><img alt="" src="images/cover.png" style="height: 360px;"/></center>

<center><h2><strong><font color="blue">DEDS-03: Data Acquisition I – APIs & Automation</font></strong></h2></center>

<b><center><h3>(C) Taufik Sutanto</h3></center>

<center><h2><strong><font color="blue">Outline</font></strong></h2></center>

* Understanding REST APIs (GET, POST, Headers, Auth).
* Authentication methods (API Keys, OAuth basics).
* Rate limiting and error handling in scripts

**Exercise/Assignment**
> Write a Python script to fetch data from a public API (e.g., Twitter/X, OpenWeather, or Semantic Scholar) and save it to JSON/CSV.

<center><h2><strong><font color="blue">Prologue</font></strong></h2></center>

<center><img alt="" src="images/deds-03/Data-Source-Primary-Secondary.png" style="height: 420px;"/></center>

<center><h2><strong><font color="blue">The Rationale for REST APIs and Automation in Research</font></strong></h2></center>

Before delving into technical implementation, researchers must understand why Application Programming Interfaces (APIs) are indispensable for modern data science. This module explores the shift from manual data collection to automated pipelines.

1. **The Transition: Manual vs. Automated Acquisition**
Traditionally, researchers relied on "static" data collection—downloading CSV or Excel files manually from websites. In a production-grade environment, this approach presents several critical failures:
* **Human Error**: Manual downloads are prone to inconsistencies and lack of version control.
* **Data Stale-ness**: Research based on static files becomes obsolete as soon as the source updates.
* **Scalability**: It is impossible to manually download millions of data points across thousands of sub-pages.
Automation via APIs resolves these issues by allowing Python scripts to communicate directly with data providers, ensuring your research is reproducible, real-time, and scalable.

2. **Real-World Case Studies**
* **Case A: Public Health Monitoring (COVID-19)**
During the pandemic, static reports were insufficient. Researchers utilized the Johns Hopkins University API to pull daily infection rates globally. Automation allowed for real-time dashboard updates and predictive modeling that influenced policy decisions.
**Outcome**: Rapid response and high-frequency data auditing.
* **Case B: Computational Social Science (Sentiment Analysis)**
Researchers studying public opinion during elections cannot rely on static datasets. By using the X (formerly Twitter) API, scholars can stream millions of tweets to analyze shifting political sentiments in real-time.
**Outcome**: Longitudinal studies that capture granular social shifts.
* **Case C: Bibliometrics and Science Mapping**
To understand the impact of research, platforms like Semantic Scholar or OpenAlex provide APIs. Data engineers automate the extraction of citation networks to map the evolution of scientific fields.
**Outcome**: Comprehensive mapping of research trends without manual entry.

<center><img alt="" src="images/deds-03/motivation-rest-API.png" style="height: 320px;"/></center>

# Why Automation is a "Production-Grade" Requirement

Mastering APIs and automation is what differentiates a data analyst from a Data Engineer.
* **Reproducibility**: A colleague can run your script and get the exact same data structure you used.
* **Integrity**: APIs provide structured data (JSON/XML), reducing the "cleaning" overhead associated with messy Excel files.
* **Scheduling**: Using tools like Cron or Airflow (Week 13), your API scripts can run every midnight, ensuring your database is always ready for analysis.

## Academic References for Further Reading
* Lazer, D., et al. (2009). Life in the network: the coming age of computational social science. Science. (Discusses the shift toward digital breadcrumbs via APIs). https://pmc.ncbi.nlm.nih.gov/articles/PMC2745217/ 
* Glez-Peña, D., et al. (2014). Web scraping technologies in an API world. Briefings in Bioinformatics. (A comparison between scraping and API utilization).  Glez-Peña, D., et al. (2014). Web scraping technologies in an API world. Briefings in Bioinformatics. https://pubmed.ncbi.nlm.nih.gov/23632294/ 
* The Turing Way Community. Guide for Reproducible Research. (An open-source handbook on why automation is critical for scientific integrity). https://book.the-turing-way.org/

> ***Without APIs, data science is limited to "one-off" snapshots. With APIs and automation, your research becomes a living system capable of evolving alongside the world's data***.

<center><h2><strong><font color="blue">Introduction to REST APIs</font></strong></h2></center>

<center><img alt="" src="images/deds-03/understanding-rest-API.png" style="height: 420px;"/></center>

<center><h2><strong><font color="blue">Understanding REST APIs (GET, POST, Headers, and Auth)</font></strong></h2></center>

In the lifecycle of a data engineering project, data acquisition is the first critical step. While manual downloads (CSV/Excel) are common in basic analysis, production-grade research relies on APIs to automate data retrieval, ensuring your dataset remains current and reproducible.

### 1. What is a REST API?
REST (Representational State Transfer) is an architectural style for providing standards between computer systems on the web. An API (Application Programming Interface) acts as a messenger that delivers your request to a provider and then delivers the response back to you.

The Analogy:
* You (The Client): A researcher sitting at a desk (Python script).
* The API (The Waiter): Takes your order to the kitchen.
* The Server (The Kitchen): Processes the request and prepares the data.
* Data (The Meal): The JSON or XML response returned to your script.

<center><img alt="" src="images/meme-cartoon/APIServersCooksExample.jpeg" style="height: 360px;"/></center>

### 2. HTTP Methods: GET vs. POST
To interact with an API, we use HTTP verbs to tell the server what action we want to perform.

**A. The GET Method (Retrieving Data)**
The most common method for researchers. Use GET when you want to request data from a specific resource.
Example: Fetching the last 100 tweets from a specific user.
Characteristics: Data is sent in the URL (query parameters). It should never change the state of the server.

**B. The POST Method (Sending Data)**
Use POST when you want to submit data to be processed to a specific resource.
Example: Sending a text snippet to a Sentiment Analysis API to get a score.
Characteristics: Data is sent in the "body" of the request, not the URL.

### 3. Request Headers: The Metadata
Headers allow the client and the server to pass additional information with an HTTP request or response. For a Data Engineer, headers are crucial for defining what kind of data we expect.
* Content-Type: Tells the server what format we are sending (e.g., application/json).
* Accept: Tells the server what format we want back.
* User-Agent: Identifies your script to the server (helps prevent being blocked as a bot).

<center><h2><strong><font color="blue">Authentication (Auth): The Digital Key</font></strong></h2></center>

Most professional research APIs are not completely "open." They require Authentication to track usage and prevent abuse. 

### A. API Keys
A simple string (token) that identifies you. Usually passed in the header.
```python
# Conceptual Header Example
headers = {
    "X-API-KEY": "your_secret_key_here"
}
```
### OAuth (Bearer Tokens)
A more secure method used by platforms like X (Twitter) or Google. You exchange credentials for a temporary "Bearer Token."

<center><img alt="" src="images/deds-03/api-vs-outh.png" style="height: 420px;"/></center>

<center><h2><strong><font color="blue">Practical Python Example</font></strong></h2></center>

For the first example we use the requests library in Python to interact with REST APIs.

Case Study: Fetching Public User Data

In [41]:
import requests

# 1. Define the Endpoint (The URL)
url = "https://api.github.com/users/taufiksutanto"

# 2. Define Headers (Best Practice)
headers = {
    "Accept": "application/vnd.github.v3+json",
    "User-Agent": "UIII-Data-Engineering-Course"
}

# 3. Execute the GET Request
response = requests.get(url, headers=headers)

# 4. Handle the Response
if response.status_code == 200:
    data = response.json()  # Convert JSON response to Python Dictionary
    #print(f"User Name: {data['name']}")
    #print(f"Location: {data['location']}")
else:
    print(f"Error: {response.status_code}")

In [42]:
# Full Data Json Format - it's a Dictionary in Python
data

{'login': 'taufiksutanto',
 'id': 8521615,
 'node_id': 'MDQ6VXNlcjg1MjE2MTU=',
 'avatar_url': 'https://avatars.githubusercontent.com/u/8521615?v=4',
 'gravatar_id': '',
 'url': 'https://api.github.com/users/taufiksutanto',
 'html_url': 'https://github.com/taufiksutanto',
 'followers_url': 'https://api.github.com/users/taufiksutanto/followers',
 'following_url': 'https://api.github.com/users/taufiksutanto/following{/other_user}',
 'gists_url': 'https://api.github.com/users/taufiksutanto/gists{/gist_id}',
 'starred_url': 'https://api.github.com/users/taufiksutanto/starred{/owner}{/repo}',
 'subscriptions_url': 'https://api.github.com/users/taufiksutanto/subscriptions',
 'organizations_url': 'https://api.github.com/users/taufiksutanto/orgs',
 'repos_url': 'https://api.github.com/users/taufiksutanto/repos',
 'events_url': 'https://api.github.com/users/taufiksutanto/events{/privacy}',
 'received_events_url': 'https://api.github.com/users/taufiksutanto/received_events',
 'type': 'User',
 'us

## 2nd Example - Web Scraping

In [6]:
!pip install beautifulsoup4 lxml --q

In [10]:
# Server tidak suka request tanpa "header"/identifier
import warnings; warnings.simplefilter('ignore')
import urllib.request
from bs4 import BeautifulSoup as bs

h = {'User-Agent' : "Mozilla/5.0 (X11; Linux i686) AppleWebKit/537.17 (KHTML, like Gecko) Chrome/24.0.1312.27 Safari/537.17"}
url = 'https://uiii.ac.id/'

# 1. Create the Request object
req = urllib.request.Request(url, headers=h)

# 2. Pass the object to urlopen
doc = urllib.request.urlopen(req).read()
web_data = bs(doc,'html.parser').text
print(web_data)

Home - Universitas Islam Internasional Indonesia             
      Skip to contentJusuf Kalla LibraryJDIHStudentsUIII Endowment FundCareersPPID UIIICMHS
Jusuf Kalla LibraryJDIHStudentsUIII Endowment FundCareersPPID UIIICMHS

 



 



 



 



 



 



 AboutAbout UIIIHistoryLeadership and GovernanceBoard of TrusteesBoard of UniversityVision & Mission MilestoneStudent Honor PledgeAcademicsAcademic ProgramsAcademic CalendarDegree ProgramsUIII Dual Degree ProgramsUIII – Deakin University Dual Degree ProgramUIII – SOAS University of London, UK Dual Degree ProgramUIII – University of Dundee, UK Dual Degree ProgramDual Master’s Degree Program UIII – The University of EdinburghFaculty and ProgramsFaculty of Islamic StudiesFaculty of Social SciencesFaculty of Economics and BusinessFaculty of EducationScience & TechnologyCurriculumTeaching and Learning EnvironmentAdmissionsHow to ApplyLanguage RequirementsResearchResearch at UIIIResearch Grant ProgramCentersFind a ResearcherUIII Scholar Cen

### Raw Data before parsing

In [11]:
bs(doc,'html.parser')

<!DOCTYPE html>
<html lang="en-US"><head><meta charset="utf-8"/><meta content="width=device-width, initial-scale=1" name="viewport"/><link href="https://gmpg.org/xfn/11" rel="profile"/><meta content="index, follow, max-image-preview:large, max-snippet:-1, max-video-preview:-1" name="robots"><title>Home - Universitas Islam Internasional Indonesia</title><meta content="Creating a better world through excellent graduate education and research on Islam and society in the Muslim world" name="description"><link href="https://uiii.ac.id/" rel="canonical"><meta content="en_US" property="og:locale"/><meta content="website" property="og:type"/><meta content="Home" property="og:title"/><meta content="Learn about Universitas Islam Internasional Indonesia (UIII) and its diverse academic programs focused on Islam and the Muslim world." property="og:description"/><meta content="https://uiii.ac.id/" property="og:url"/><meta content="Universitas Islam Internasional Indonesia" property="og:site_name"/><

# 3rd Example Using Pandas to Get Tables of Data From (Web)Sites

In [22]:
import pandas as pd
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/Health_in_Indonesia"

session = requests.Session()
session.headers.update(h)
response = session.get(url)
response.raise_for_status()
html = StringIO(response.text)

tables = pd.read_html(html, flavor="lxml")
print(f"Found {len(tables)} tables")
df = tables[0]
df.head()

Found 11 tables
      Period  Life expectancy in years   Period.1  Life expectancy in years.1
0  1950–1955                     41.75  1985–1990                       61.29
1  1955–1960                     45.07  1990–1995                       63.32
2  1960–1965                     48.18  1995–2000                       65.16
3  1965–1970                     51.05  2000–2005                       66.40
4  1970–1975                     54.02  2005–2010                       68.31


In [32]:
tables[1].head()

,Observed deaths per 1000 live births,1990,2000,2010,2019
0,Under 28 days,30.6,22.8,17.4,12.4
1,Under 1 year,61.8,41.0,28.0,20.2
2,Under 5 years,84.0,52.2,33.9,23.9


### Saving Data(frames) 

In [35]:
filename = "data/indonesian-health-data"
for i, table in enumerate(tables):
    table.to_csv(filename+"-{}.csv".format(i), index=False)
"Done"

'Done'

<center><h2><strong><font color="blue">Authentication Methods (API Keys & OAuth Basics)</font></strong></h2></center>

<center><img alt="" src="images/deds-03/api-oauth-authentication.jpg" style="height: 420px;"/></center>

<center><h2><strong><font color="blue">Acquiring Youtube API Key from Google Cloud Platform</font></strong></h2></center>

1. Register to GCP: https://cloud.google.com/  (use your gmail Account, notes that Google now enforce 2-step verification)
2. Create a Project from the "**Google Cloud Console Dashboard**" https://console.cloud.google.com/apis/dashboard
3. **Enable** Youtube Data 3 API from search bar OR click the following link: https://console.cloud.google.com/marketplace/product/google/youtube.googleapis.com?q=search&referrer=search&project=weblogin-424102
4. Go Back to https://console.cloud.google.com/apis/dashboard and Click "**Credentials**" from the Left Menu
5. Choose "**Create Credentials**" and then choose "**API Key**" from the drop down menu
6. Name your API Key (e.g. UIII-DER)
7. From the "APIs that can be accessed using this key" drop down menu, enable/choose "**Youtube Data API v3**"
8. Leave the rest of the options as it is, and then click "**Create**"
9. From the list of "API Keys" in the dashboard, choose/click your new API Key (i.e. "**UIII-DER**")
10. Click "**Show key**" from the menu on the Right-Side and store/save your **API Key** somewhere safe. Close the pop-up message box when you are finished.

<center><img alt="" src="images/deds-03/Youtube-APIkeys.png" style="height: 420px;"/></center>

## 4th Example Getting Data from Youtube using API Key - Search Videos

In [24]:
# Use your own API Key
API_KEY = ""
"Done, API Key Stored in the Memory"

'Done, API Key Stored in the Memory'

In [27]:
import requests

SEARCH_URL = "https://www.googleapis.com/youtube/v3/search"
def search_videos(query, max_results=5):
    params = {
        "part": "snippet",
        "q": query,
        "type": "video",
        "maxResults": max_results,
        "key": API_KEY
    }

    response = requests.get(SEARCH_URL, params=params)
    data = response.json()
    allData = []
    for item in data.get("items", []):
        video_id = item["id"]["videoId"]
        title = item["snippet"]["title"]
        channel = item["snippet"]["channelTitle"]
        print(f"Title   : {title}")
        print(f"Channel : {channel}")
        print(f"URL     : https://www.youtube.com/watch?v={video_id}")
        print("-" * 40)
        allData.append(item)
    return allData

# Example usage
allData = search_videos("data engineering in health")

Title   : What Does a Data Engineer ACTUALLY Do?
Channel : Learn with Lukas
URL     : https://www.youtube.com/watch?v=hTjo-QVWcK0
----------------------------------------
Title   : 4 Types of Healthcare Data Analysts Should Know
Channel : Data Wizardry
URL     : https://www.youtube.com/watch?v=4yFgYD-RYBU
----------------------------------------
Title   : Data Engineering in Healthcare Domain
Channel : Harbinger Group
URL     : https://www.youtube.com/watch?v=VCjiM3ex9Kg
----------------------------------------
Title   : Data Engineers vs Data Analysts vs Data Scientists | What&#39;s right for you?
Channel : Mo Chen
URL     : https://www.youtube.com/watch?v=eUrwxH9as_k
----------------------------------------
Title   : End-to-End Healthcare Data Engineer Project | Azure Databricks + Data Factory
Channel : Data with Jay
URL     : https://www.youtube.com/watch?v=01LVHch-1x0
----------------------------------------


In [28]:
# Full Json Data for the first video
allData[0]

{'kind': 'youtube#searchResult',
 'etag': '4C8uftgjT-gsYnhto7oybRY5X8s',
 'id': {'kind': 'youtube#video', 'videoId': 'hTjo-QVWcK0'},
 'snippet': {'publishedAt': '2023-11-07T11:57:18Z',
  'channelId': 'UCw_LFe2pS8x3NyipGNJgeEA',
  'title': 'What Does a Data Engineer ACTUALLY Do?',
  'description': 'Free Data Analyst Roadmap: https://learnwithlukas.com/da Data Analyst Certification: https://datacamp.pxf.io/lukasanalyst ...',
  'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/hTjo-QVWcK0/default.jpg',
    'width': 120,
    'height': 90},
   'medium': {'url': 'https://i.ytimg.com/vi/hTjo-QVWcK0/mqdefault.jpg',
    'width': 320,
    'height': 180},
   'high': {'url': 'https://i.ytimg.com/vi/hTjo-QVWcK0/hqdefault.jpg',
    'width': 480,
    'height': 360}},
  'channelTitle': 'Learn with Lukas',
  'liveBroadcastContent': 'none',
  'publishTime': '2023-11-07T11:57:18Z'}}

### Saving json Data

In [39]:
import json

fileName = "data/youtube-search-data.json"
# Save to a JSON file
with open(fileName, "w") as f:
    json.dump(allData, f, indent=4)

print("Data successfully saved")

Data successfully saved


### import the saved Json Data

In [40]:
with open(fileName, "r") as f:
    loaded_data = json.load(f)

print(loaded_data[0])

{'kind': 'youtube#searchResult', 'etag': '4C8uftgjT-gsYnhto7oybRY5X8s', 'id': {'kind': 'youtube#video', 'videoId': 'hTjo-QVWcK0'}, 'snippet': {'publishedAt': '2023-11-07T11:57:18Z', 'channelId': 'UCw_LFe2pS8x3NyipGNJgeEA', 'title': 'What Does a Data Engineer ACTUALLY Do?', 'description': 'Free Data Analyst Roadmap: https://learnwithlukas.com/da Data Analyst Certification: https://datacamp.pxf.io/lukasanalyst ...', 'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/hTjo-QVWcK0/default.jpg', 'width': 120, 'height': 90}, 'medium': {'url': 'https://i.ytimg.com/vi/hTjo-QVWcK0/mqdefault.jpg', 'width': 320, 'height': 180}, 'high': {'url': 'https://i.ytimg.com/vi/hTjo-QVWcK0/hqdefault.jpg', 'width': 480, 'height': 360}}, 'channelTitle': 'Learn with Lukas', 'liveBroadcastContent': 'none', 'publishTime': '2023-11-07T11:57:18Z'}}


## 5th Example Getting Data from Youtube using API Key - get Video Details

In [30]:
VIDEO_URL = "https://www.googleapis.com/youtube/v3/videos"

def get_video_details(video_id):
    params = {
        "part": "snippet,statistics",
        "id": video_id,
        "key": API_KEY
    }

    response = requests.get(VIDEO_URL, params=params)
    data = response.json()

    item = data["items"][0]

    title = item["snippet"]["title"]
    views = item["statistics"].get("viewCount")
    likes = item["statistics"].get("likeCount")

    print(f"Title : {title}")
    print(f"Views : {views}")
    print(f"Likes : {likes}")
    return item

# Example usage
item = get_video_details("hTjo-QVWcK0")

Title : What Does a Data Engineer ACTUALLY Do?
Views : 186123
Likes : 2927


In [36]:
# Full Data
item

{'kind': 'youtube#video',
 'etag': 'cDLeyRGDm0gPeruy2qXr-FVxDOo',
 'id': 'hTjo-QVWcK0',
 'snippet': {'publishedAt': '2023-11-07T11:57:18Z',
  'channelId': 'UCw_LFe2pS8x3NyipGNJgeEA',
  'title': 'What Does a Data Engineer ACTUALLY Do?',
  'description': '✅ Free Data Analyst Roadmap: https://learnwithlukas.com/da\n🚀 Data Analyst Certification: https://datacamp.pxf.io/lukasanalyst\n🚀 SQL Certification: https://datacamp.pxf.io/sqlcertificate\n🚀 Power BI Certificate: https://imp.i384100.net/poweranalyst\n\n▬▬▬▬\n\nThe information on this YouTube Channel and the resources available are for educational and informational purposes only. As an affiliate, I may earn when you sign up to websites/ for services provided in the description. This allows me to run the channel and make videos for you. Any income or earnings representations are aspirational statements only of your earning potential. There are no guarantees that you’ll receive the same results or any results at all. I am not a financial o

<center><h2><strong><font color="blue">Rate limiting and error handling in scripts</font></strong></h2></center>

* GCP~Youtube Data API Documentation: https://developers.google.com/youtube/v3/getting-started#intro

### Identifying API Errors

When a pipeline encounters an issue, the API returns an HTTP status code. Data engineers must program the script to interpret these codes appropriately.
* 200 OK: Successful operation.
* 400 Bad Request: The script submitted invalid parameters.
* 403 Forbidden (Quota Exceeded): The daily limit is depleted. The script must halt execution until the quota resets (typically at midnight Pacific Time).
* 429 Too Many Requests: The script is transmitting requests too rapidly.
* 500 / 503 Server Error: The YouTube server is temporarily unavailable.

<center><img alt="" src="images/deds-03/api-quota-error-response-code.jpg" style="height: 420px;"/></center>

<center><h2><strong><font color="blue">Other Open/Free APIs</font></strong></h2></center>

### You can use this data sources for your first assignment 

| No | Data Source                  | URL                                                                 | Quota (Free Tier)                                                                 | Link of the API Provider                                      |
|----|------------------------------|----------------------------------------------------------------------|------------------------------------------------------------------------------------|----------------------------------------------------------------|
| 1  | YouTube Data API v3          | https://www.googleapis.com/youtube/v3                               | 10,000 units/day                                                                   | https://developers.google.com/youtube/v3                      |
| 2  | Twitter (X) API              | https://api.twitter.com/2/                                          | Limited (e.g., ~1,500 tweets/month for free tier, subject to change)               | https://developer.x.com/en/docs/twitter-api                  |
| 3  | Reddit API                   | https://www.reddit.com/dev/api/                                     | ~100 queries/minute                                                                | https://www.reddit.com/dev/api                               |
| 4  | OpenWeatherMap API           | https://api.openweathermap.org                                     | 1,000 calls/day (free tier)                                                        | https://openweathermap.org/api                               |
| 5  | World Bank Open Data API     | https://api.worldbank.org/v2                                        | No strict limit (fair usage applies)                                               | https://datahelpdesk.worldbank.org/knowledgebase/topics/125589|
| 6  | NASA Open APIs               | https://api.nasa.gov                                                | 1,000 requests/hour (DEMO_KEY)                                                     | https://api.nasa.gov                                         |
| 7  | OpenAQ (Air Quality)         | https://api.openaq.org                                              | No strict limit (fair use)                                                         | https://docs.openaq.org                                      |
| 8  | NewsAPI                      | https://newsapi.org/v2                                              | 100 requests/day                                                                   | https://newsapi.org/docs                                    |
| 9  | GitHub REST API              | https://api.github.com                                              | 60 requests/hour (unauthenticated), 5,000/hour (authenticated)                     | https://docs.github.com/en/rest                             |
| 10 | OpenStreetMap (Overpass API) | https://overpass-api.de/api/interpreter                             | Rate-limited (varies, ~10,000 queries/day typical fair use)                        | https://wiki.openstreetmap.org/wiki/Overpass_API            |
| 11 | CoinGecko API (Crypto)       | https://api.coingecko.com/api/v3                                    | ~10-50 calls/minute                                                                | https://www.coingecko.com/en/api/documentation              |
| 12 | ExchangeRate API             | https://api.exchangerate.host                                       | No API key required, fair usage                                                    | https://exchangerate.host/#/                                |
| 13 | Kaggle API                   | https://www.kaggle.com/api/v1                                       | No strict limit (authentication required)                                          | https://www.kaggle.com/docs/api                             |
| 14 | Spotify Web API              | https://api.spotify.com/v1                                          | Rate-limited (varies; approx. 100 requests/30 seconds per user token)              | https://developer.spotify.com/documentation/web-api         |
| 15 | TMDB (Movie DB) API          | https://api.themoviedb.org/3                                        | ~40 requests/10 seconds                                                            | https://developer.themoviedb.org/docs                       |

<center><h2><strong><font color="blue">End of Module</font></strong></h2></center>

<center><video width="256" controls src="videos/Cat-Rest-API.mp4"></video></center>

* Source: https://www.tiktok.com/@coding.kitty7/video/7612591987782356257